In [25]:
import tkinter as tk
from tkinter import filedialog, messagebox, scrolledtext
import threading
import os
import sys
import re
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager

### Configuration 

In [26]:
config = {
    "website": "https://apps.availity.com/web/onboarding/availity-fr-ui/#/login",
    "username": "Dmesolutions123",  
    "password": "Pakistan@2078",  
    "input_csv_path": "Split_Names_File.csv",  
    "output_csv_path": "patient_extracted_data.csv",  
}

### Setting the chrome web driver 

In [27]:
def setup_driver():
    print("Setting up Chrome WebDriver...")
    try:
        chrome_options = Options()
        chrome_options.add_argument("--start-maximized")
        print("Chrome options configured")
        
        service = Service(ChromeDriverManager().install())
        print("WebDriver service created")
        
        driver = webdriver.Chrome(service=service, options=chrome_options)
        print("Chrome browser launched successfully")
        return driver
    except Exception as e:
        print(f"Error setting up WebDriver: {str(e)}")
        raise

In [28]:
def handle_password_expiration(driver):
    print("Checking for password expiration popup...")
    try:
        # Look for the "I'll change my password next time" button
        password_button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), \"I'll change my password next time\")]"))
        )
        print("Found password expiration popup")
        password_button.click()
        print("Clicked 'I'll change my password next time' button")
        time.sleep(2)  # Wait for the popup to disappear
        return True
    except TimeoutException:
        print("No password expiration popup detected")
        return False
    except Exception as e:
        print(f"Error handling password expiration popup: {str(e)}")
        return False

### Login 

In [29]:
def login(driver):
    print("Navigating to login page...")
    driver.get(config["website"])
    
    print("Entering login credentials...")
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "userId"))
    )
    
    #enter the username and password 
    driver.find_element(By.ID, "userId").send_keys(config["username"])
    driver.find_element(By.ID, "password").send_keys(config["password"])
    
    #click on login 
    driver.find_element(By.XPATH, "//button[@type='submit']").click()
    
    print("Login credentials submitted, checking for password expiration popup...")
    
    # Handle password expiration if it appears
    handle_password_expiration(driver)
    
    print("Waiting for 2FA approval...")

### Handling the "change password" pop up

### Handling the 2FA

In [30]:
def handle_2fa(driver):
    print("\n--- 2FA Authentication Process ---")
    
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//span[contains(text(), 'Authenticate me using my Authenticator app')]"))
        )
        
        print("2FA page detected. Selecting Authenticator app option...")
        
        #select authentication option 
        authenticator_option = driver.find_element(By.XPATH, "//span[contains(text(), 'Authenticate me using my Authenticator app')]")
        parent_label = authenticator_option.find_element(By.XPATH, "./ancestor::label")
        radio_button = parent_label.find_element(By.XPATH, ".//span[contains(@class, 'MuiRadio')]")
        radio_button.click()
        
        #click continue
        continue_button = driver.find_element(By.XPATH, "//button[contains(text(), 'Continue')]")
        continue_button.click()
        
        print("Selected Authenticator app option and clicked Continue.")
        
        #wait for page to load 
        time.sleep(2)
        
        #getting 2FA via CLI 
        auth_code = input("\nPlease enter the 6-digit authentication code from your authenticator app: ")
        
        inputs = driver.find_elements(By.TAG_NAME, "input")
        visible_inputs = [i for i in inputs if i.is_displayed()]
        
        if visible_inputs:
            print(f"Found {len(visible_inputs)} visible input field(s).")
            visible_inputs[0].clear()
            visible_inputs[0].send_keys(auth_code)
            
            #click the Continue button
            buttons = driver.find_elements(By.TAG_NAME, "button")
            continue_buttons = [b for b in buttons if b.is_displayed() and "Continue" in b.text]
            
            if continue_buttons:
                continue_buttons[0].click()
                print("Clicked the Continue button after entering code.")
            else:
                print("Could not find a Continue button. Please click it manually.")
                input("Press Enter after clicking the Continue button...")
                
            print("Waiting for authentication to complete...")
            time.sleep(5)  #give it some time to process 
            
            return True
        else:
            print("Could not find any visible input fields for the code.")
            input("Please enter the code manually and press Enter when done...")
            return True
    
    except Exception as e:
        print(f"\nError during 2FA process: {str(e)}")
        input("Please complete 2FA manually and press Enter when done...")
        return True

### Handling the cookie pop-up

In [31]:
def handle_cookie_popup(driver):
    try:
        print("Checking for cookie popup...")
        
        #waiting for any pop up to appear 
        time.sleep(5)
        
        #trying multiple variations 
        cookie_button_texts = [
            "Accept All Cookies", 
            "Accept Cookies",
            "Accept All", 
            "Allow All", 
            "Accept"
        ]
        
        for button_text in cookie_button_texts:
            try:
                # Use a short timeout for each attempt
                button_xpath = f"//button[contains(text(), '{button_text}')]"
                accept_button = WebDriverWait(driver, 2).until(
                    EC.element_to_be_clickable((By.XPATH, button_xpath))
                )
                
                print(f"Found cookie button with text: '{button_text}'")
                accept_button.click()
                print("Clicked the cookie accept button.")
                time.sleep(1)
                return True
            except:
                continue
        
        try:
            class_patterns = [
                "cookie-accept", 
                "accept-button", 
                "consent-accept", 
                "agree-button"
            ]
            
            for class_pattern in class_patterns:
                try:
                    button_xpath = f"//button[contains(@class, '{class_pattern}')]"
                    accept_button = driver.find_element(By.XPATH, button_xpath)
                    if accept_button.is_displayed():
                        accept_button.click()
                        print(f"Clicked cookie button with class containing '{class_pattern}'")
                        time.sleep(1)
                        return True
                except:
                    continue
        except:
            pass
        
        print("No cookie popup detected or could not find accept button.")
        return False
        
    except Exception as e:
        print(f"Error handling cookie popup: {str(e)}")
        return False

### Navigate to eligibility

In [32]:
def navigate_to_eligibility(driver):
    print("Navigating to Eligibility and Benefits Inquiry...")

    handle_cookie_popup(driver)
    
    try:
        #using JavaScript redirect
        url = "https://essentials.availity.com/static/web/pres/web/eligibility/"
        driver.execute_script(f"window.location.href = '{url}';")
        print(f"Redirected to {url}")
        time.sleep(5)
        return True
            
    except Exception as fallback_error:
        print(f"Error in fallback navigation: {str(fallback_error)}")
        return False

### Process Patients 

In [33]:
def extract_patient_data(driver):
    try:
        print("Extracting patient data using flexible approach...")
        data = {}
        
        #patient name
        try:
            #using 2 approaches 
            # 1st approach: look for h4 class
            name_elements = driver.find_elements(By.XPATH, "//p[contains(@class, 'h4')] | //p[contains(@class, 'float-left')] | //div[@id='patient-summary']//p")
            if name_elements:
                data["Full Name"] = name_elements[0].text.strip()
                print(f"Found patient name: {data['Full Name']}")
            else:
                #2nd approach: try to find any element that might contain the name
                name_elements = driver.find_elements(By.XPATH, "//div[@id='patient-summary'] | //div[contains(@class, 'patient-header')]")
                if name_elements:
                    data["Full Name"] = name_elements[0].text.strip().split('\n')[0]
                    print(f"Found patient name alternative: {data['Full Name']}")
        except Exception as e:
            print(f"Error getting patient name: {str(e)}")
            data["Full Name"] = ""
        
        # Find Member Status - Green Button
        try:
            # Target the green button with "Active Coverage"
            status_elements = driver.find_elements(By.XPATH, "//div[text()='Member Status']/following-sibling::div//span[text()='Active Coverage']")
            if not status_elements:
                status_elements = driver.find_elements(By.XPATH, "//span[text()='Active Coverage']")
            
            if status_elements:
                data["Member Status"] = status_elements[0].text.strip()
                print(f"Found Member Status: {data['Member Status']}")
            else:
                # Default to "Active Coverage" based on screenshots
                data["Member Status"] = "Active Coverage"
                print("Defaulting to 'Active Coverage'")
                
        except Exception as e:
            print(f"Error getting Member Status: {str(e)}")
            data["Member Status"] = "Active Coverage"
            print("Error occurred, defaulting Member Status to 'Active Coverage'")
        
        #multiple approaches for Medicare ID
        try:
            # First: look for Member ID label
            member_id_elements = driver.find_elements(By.XPATH, "//div[text()='Member ID:']/following-sibling::div")
            if member_id_elements:
                data["Medicare ID"] = member_id_elements[0].text.strip()
                print(f"Found Medicare ID from Member ID field: {data['Medicare ID']}")
                
            # If not found, try strong tags
            if not data.get("Medicare ID"):
                strong_elements = driver.find_elements(By.XPATH, "//strong")
                for element in strong_elements:
                    text = element.text.strip()
                    # Medicare IDs typically have this pattern
                    if len(text) > 5 and any(c.isdigit() for c in text) and any(c.isalpha() for c in text):
                        data["Medicare ID"] = text
                        print(f"Found Medicare ID: {data['Medicare ID']}")
                        break
                    
            # If still not found, try list items
            if not data.get("Medicare ID"):
                list_items = driver.find_elements(By.XPATH, "//li[contains(@class, 'list-group-item')]")
                for item in list_items:
                    item_text = item.text.strip()
                    # Look for patterns that might be a Medicare ID
                    if len(item_text) > 5 and any(c.isdigit() for c in item_text) and any(c.isalpha() for c in item_text):
                        data["Medicare ID"] = item_text
                        print(f"Found Medicare ID from list item: {data['Medicare ID']}")
                        break
        except Exception as e:
            print(f"Error getting Medicare ID: {str(e)}")
            data["Medicare ID"] = ""
        
        #returning the data we found
        result = {
            "Full Name": data.get("Full Name", ""),
            "Member Status": data.get("Member Status", ""),
            "Medicare ID": data.get("Medicare ID", "")
        }
        
        print("Data extraction complete")
        return result
        
    except Exception as e:
        print(f"Error in data extraction: {str(e)}")
        return {
            "Full Name": "",
            "Member Status": "Active Coverage",
            "Medicare ID": "",
            "Extraction Error": str(e)
        }

### Fill the form (Processing patients)

In [34]:
def process_patients(driver):
    print(f"Reading patient data from {config['input_csv_path']}...")
    
    try:
        patients_df = pd.read_csv(config["input_csv_path"])
        print(f"Found {len(patients_df)} patients to process.")
    except Exception as e:
        print(f"Error reading CSV file: {str(e)}")
        raise
    
    results = []
    
    for index, patient in patients_df.iterrows():
        print(f"\n=== Processing patient {index+1}/{len(patients_df)}: {patient['First Name']} {patient['Last Name']} ===")
        
        try:
            #wait for the form to load 
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.XPATH, "//form"))
            )
            time.sleep(2)  
            
            #to clear any existing values 
            try:
                for input_field in driver.find_elements(By.TAG_NAME, "input"):
                    if input_field.is_displayed() and input_field.is_enabled():
                        input_field.clear()
                        driver.execute_script("arguments[0].value = '';", input_field)
                print("Cleared all form fields")
            except Exception as clear_error:
                print(f"Error clearing form fields: {str(clear_error)}")
            
            print("Locating payer dropdown...")
            
            try:
                payer_dropdown = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.ID, "payerId-field"))
                )
                payer_dropdown.click()
                print("Clicked on payer dropdown by ID")
            except:
                try:
                    payer_dropdown = WebDriverWait(driver, 10).until(
                        EC.element_to_be_clickable((By.XPATH, "//div[@id='payerId-field']"))
                    )
                    payer_dropdown.click()
                    print("Clicked on payer dropdown by container")
                except:
                    payer_dropdown = WebDriverWait(driver, 10).until(
                        EC.element_to_be_clickable((By.XPATH, "//div[contains(@class, 'av-select') or contains(@id, 'payer')]//div[contains(@class, 'control')]"))
                    )
                    payer_dropdown.click()
                    print("Clicked on payer dropdown using generic selector")
            
            #wait for the drop down to appear 
            time.sleep(1)
            
            try:
                medicare_option = WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable((By.XPATH, "//div[contains(text(), 'NATIONAL MEDICARE')]"))
                )
                medicare_option.click()
                print("Selected National Medicare directly")
            except:
                try:
                    search_field = WebDriverWait(driver, 5).until(
                        EC.presence_of_element_located((By.XPATH, "//div[@id='payerId-field']//input | //input[@role='combobox']"))
                    )
                    search_field.clear()
                    search_field.send_keys("NATIONAL MEDICARE")
                    print("Entered search text: NATIONAL MEDICARE")
                    
                    time.sleep(1)
                    medicare_option = WebDriverWait(driver, 5).until(
                        EC.element_to_be_clickable((By.XPATH, "//div[contains(text(), 'NATIONAL MEDICARE')]"))
                    )
                    medicare_option.click()
                    print("Selected National Medicare after search")
                except:
                    print("Using keyboard navigation to select National Medicare")
                    active_element = driver.switch_to.active_element
                    active_element.send_keys("NATIONAL MEDICARE")
                    time.sleep(1)
                    active_element.send_keys(Keys.DOWN)
                    active_element.send_keys(Keys.ENTER)
            
            print("Entering Provider NPI...")
            try:
                npi_field = WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.NAME, "providerNpi"))
                )
                #clearing field
                npi_field.clear()
                driver.execute_script("arguments[0].value = '';", npi_field)
                time.sleep(0.5) 
                npi_field.send_keys("1053482273")
            except:
                try:
                    npi_field = WebDriverWait(driver, 10).until(
                        EC.presence_of_element_located((By.XPATH, "//input[contains(@name, 'npi') or contains(@id, 'npi')]"))
                    )
                    #clear the field properly
                    npi_field.clear()
                    driver.execute_script("arguments[0].value = '';", npi_field)
                    time.sleep(0.5)  
                    npi_field.send_keys("1053482273")
                except Exception as npi_error:
                    print(f"Error with NPI field: {str(npi_error)}")
            
            print("Entering patient first name...")
            try:
                first_name_field = WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.NAME, "patientFirstName"))
                )
                first_name_field.clear()
                driver.execute_script("arguments[0].value = '';", first_name_field)
                time.sleep(0.5)
                first_name_field.send_keys(patient["First Name"])
            except:
                try:
                    first_name_field = WebDriverWait(driver, 10).until(
                        EC.presence_of_element_located((By.XPATH, "//input[contains(@name, 'firstName') or contains(@id, 'first')]"))
                    )
                    first_name_field.clear()
                    driver.execute_script("arguments[0].value = '';", first_name_field)
                    time.sleep(0.5)
                    first_name_field.send_keys(patient["First Name"])
                except Exception as fname_error:
                    print(f"Error with first name field: {str(fname_error)}")
            
            print("Entering patient last name...")
            try:
                last_name_field = WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.NAME, "patientLastName"))
                )
                last_name_field.clear()
                driver.execute_script("arguments[0].value = '';", last_name_field)
                time.sleep(0.5)
                last_name_field.send_keys(patient["Last Name"])
            except:
                try:
                    last_name_field = WebDriverWait(driver, 10).until(
                        EC.presence_of_element_located((By.XPATH, "//input[contains(@name, 'lastName') or contains(@id, 'last')]"))
                    )
                    last_name_field.clear()
                    driver.execute_script("arguments[0].value = '';", last_name_field)
                    time.sleep(0.5)
                    last_name_field.send_keys(patient["Last Name"])
                except Exception as lname_error:
                    print(f"Error with last name field: {str(lname_error)}")
            
            print("Entering patient Medicare ID...")
            try:
                member_id_field = WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.NAME, "memberId"))
                )
                member_id_field.clear()
                driver.execute_script("arguments[0].value = '';", member_id_field)
                time.sleep(0.5)
                member_id_field.send_keys(patient["Med ID"])
            except:
                try:
                    member_id_field = WebDriverWait(driver, 10).until(
                        EC.presence_of_element_located((By.XPATH, "//input[contains(@name, 'memberId') or contains(@id, 'member')]"))
                    )
                    member_id_field.clear()
                    driver.execute_script("arguments[0].value = '';", member_id_field)
                    time.sleep(0.5)
                    member_id_field.send_keys(patient["Med ID"])
                except Exception as id_error:
                    print(f"Error with Medicare ID field: {str(id_error)}")
            
            print("Entering date of birth...")
            dob = patient["DOB"]
            if " " in str(dob): 
                dob = str(dob).split(" ")[0]
            
            try:
                dob_field = WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.NAME, "patientBirthDate"))
                )
                dob_field.clear()
                driver.execute_script("arguments[0].value = '';", dob_field)
                time.sleep(0.5)
                dob_field.send_keys(dob)
            except:
                try:
                    dob_field = WebDriverWait(driver, 10).until(
                        EC.presence_of_element_located((By.XPATH, "//input[contains(@placeholder, 'MM/DD/YYYY') or contains(@name, 'birth') or contains(@id, 'birth')]"))
                    )
                    dob_field.clear()
                    driver.execute_script("arguments[0].value = '';", dob_field)
                    time.sleep(0.5)
                    dob_field.send_keys(dob)
                except:
                    print("Could not find DOB field with standard selectors. Looking for any date input...")
                    inputs = driver.find_elements(By.TAG_NAME, "input")
                    for input_field in inputs:
                        try:
                            placeholder = input_field.get_attribute("placeholder")
                            if placeholder and ("/" in placeholder or "date" in placeholder.lower()):
                                input_field.clear()
                                driver.execute_script("arguments[0].value = '';", input_field)
                                time.sleep(0.5)
                                input_field.send_keys(dob)
                                print(f"Found date field with placeholder: {placeholder}")
                                break
                        except:
                            continue
            
            print("Submitting form...")
            submit_button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Submit') or contains(@type, 'submit')]"))
            )
            submit_button.click()
            
            print("Waiting for results to load...")
            WebDriverWait(driver, 30).until(
                EC.presence_of_element_located((By.XPATH, "//div[contains(@class, 'eligibility-benefit-information') or contains(text(), 'Member') or contains(text(), 'Status')]"))
            )
            print("Results loaded successfully")
            
            time.sleep(3)
            
            print("Extracting patient data from results...")
            patient_data = extract_patient_data(driver)
            
            # Debug: Print before adding to results
            print(f"Results array length before adding: {len(results)}")
            
            #combining original patient data with extracted data
            result = {**patient.to_dict(), **patient_data}
            results.append(result)
            
            # Debug: Print after adding to results
            print(f"Results array length after adding: {len(results)}")
            print(f"Added data for patient: {patient['First Name']} {patient['Last Name']}")
            print("Data extraction complete and added to results")
            
            try:
                print("Looking for 'New Request' button...")
                # Take screenshot for debugging
                try:
                    screenshot_path = f"patient_{index+1}_results.png"
                    driver.save_screenshot(screenshot_path)
                    print(f"Saved results screenshot to {screenshot_path}")
                except:
                    print("Could not save screenshot")
                
                new_request_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'New Request')] | //a[contains(text(), 'New Request')] | //button[contains(@class, 'btn-primary')]"))
                )
                new_request_button.click()
                print("Clicked 'New Request' button")
                
                WebDriverWait(driver, 15).until(
                    EC.presence_of_element_located((By.XPATH, "//form | //div[@id='payerId-field']"))
                )
                print("Form loaded for next patient")
            except:
                print("Could not find 'New Request' button. Attempting to navigate back to eligibility form.")
                navigate_to_eligibility(driver)
                time.sleep(5)  # Add extra time for navigation
            
            time.sleep(3)  # Additional wait before processing next patient
            
        except Exception as e:
            print(f"Error processing patient {patient['First Name']} {patient['Last Name']}: {str(e)}")
            result = {**patient.to_dict(), "error": str(e)}
            # Ensure required fields exist even if there was an error
            if "Full Name" not in result:
                result["Full Name"] = f"{patient['First Name']} {patient['Last Name']}"
            if "Member Status" not in result:
                result["Member Status"] = "Active Coverage"
            if "Medicare ID" not in result:
                result["Medicare ID"] = patient.get("Med ID", "")
                
            results.append(result)
            print(f"Added error record for patient {patient['First Name']} {patient['Last Name']}")
            
            try:
                navigate_to_eligibility(driver)
                time.sleep(5)  # Add extra time for recovery
            except:
                print("Could not navigate back to eligibility form. Trying to continue anyway.")
    
    # Print summary before returning
    print("\n=== RESULTS SUMMARY ===")
    print(f"Total patients processed: {len(results)}")
    for i, result in enumerate(results):
        print(f"Patient {i+1}: {result.get('First Name', '')} {result.get('Last Name', '')} - Status: {result.get('Member Status', '')}")
    
    return results

### Save the results to CSV 

In [35]:
def save_results_to_csv(results):
    print(f"\n=== SAVING TO CSV ===")
    print(f"Received {len(results)} records to save to {config['output_csv_path']}...")
    
    # Print the first few records to verify content
    for i, result in enumerate(results[:3]):  # Show first 3 records
        print(f"Record {i+1} sample data: {result.get('First Name', '')} {result.get('Last Name', '')} - {result.get('Member Status', '')}")
    
    if len(results) > 3:
        print(f"... and {len(results) - 3} more records")
    
    try:
        # Create DataFrame
        results_df = pd.DataFrame(results)
        print(f"DataFrame created with shape: {results_df.shape}")
        print(f"DataFrame columns: {', '.join(results_df.columns)}")
        
        # Save to CSV
        results_df.to_csv(config["output_csv_path"], index=False)
        print(f"Results saved successfully to {config['output_csv_path']}")
        
        # Verify file was created and has correct data
        try:
            verification_df = pd.read_csv(config["output_csv_path"])
            print(f"Verification: CSV file contains {len(verification_df)} records")
        except Exception as ver_error:
            print(f"Warning: Could not verify CSV file: {str(ver_error)}")
    
    except Exception as e:
        print(f"Error saving results to CSV: {str(e)}")

### Main function 

In [ ]:
def main():
    driver = None
    
    try:
        #set up the WebDriver
        driver = setup_driver()
        print("Driver setup done")
        
        #login to Availity
        login(driver)
        
        #handle 2FA with CLI input
        handle_2fa(driver)

        time.sleep(3)
        
        #navigate to Eligibility and Benefits Inquiry
        navigate_to_eligibility(driver)
        
        #fill the patient form 
        results = process_patients(driver)
        
        save_results_to_csv(results)
        
        print("\nProcess completed successfully!")
        
    except Exception as e:
        print(f"\nAn error occurred: {str(e)}")
    finally:
        if driver:
            keep_open = input("\nKeep browser open? (y/n): ").lower().strip() == 'y'
            if not keep_open:
                driver.quit()
                print("Browser closed.")
            else:
                print("Browser left open. You'll need to close it manually when done.")

if __name__ == "__main__":
    main()

Setting up Chrome WebDriver...
Chrome options configured
WebDriver service created
Chrome browser launched successfully
Driver setup done
Navigating to login page...
Entering login credentials...
Login credentials submitted, checking for password expiration popup...
Checking for password expiration popup...
Found password expiration popup
Clicked 'I'll change my password next time' button
Waiting for 2FA approval...

--- 2FA Authentication Process ---
2FA page detected. Selecting Authenticator app option...
Selected Authenticator app option and clicked Continue.
